# FD002 — Baseline Deviation **Anomaly Score** (Colab / Drive Project)

This notebook computes an **engine_id + cycle** level `anomaly_score` (0–1) using the *baseline deviation score* method:

- Per engine, define a **baseline window**: first **N** cycles.
- For each sensor: \(\mu=\text{mean}\), \(\sigma=\text{std}+\epsilon\)
- Per cycle, compute **z-score**: \(z=(x-\mu)/\sigma\)
- Reduce to single raw score: **RMS z-score**: \(\sqrt{\text{mean}(z^2)}\)
- Map to **0–1** with **min-max** or **sigmoid**.
- Optionally smooth with rolling mean.

It also optionally merges the anomaly score with your **FD002 prediction outputs** (`best_pred_BASE.csv`) to produce a decision-support table:
`engine_id, cycle, y_true, y_pred, anomaly_score, ...`.


## 0) Install + imports

> Run this once per Colab session.


In [ ]:
!pip -q install pandas numpy

import os, json
from pathlib import Path
import numpy as np
import pandas as pd


## 1) Mount Google Drive + set project root

Set `PROJECT_ROOT` to your repo folder inside Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/jet-cube-turbofan-mvp")  # <-- change if needed
assert PROJECT_ROOT.exists(), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
print("PROJECT_ROOT =", PROJECT_ROOT)


## 2) Paths (inputs + outputs)

### You need these files available in your project tree (recommended):

**A) Normalized FD002 sensor CSVs**
- `data/processed/CMAPSS_norm_full/FD002/train_*.csv`
- `data/processed/CMAPSS_norm_full/FD002/test_*.csv`

**B) Model output files (the ones you shared)**
Put them in a single folder, e.g.
- `outputs/fd002_BASE/config_BASE.json`
- `outputs/fd002_BASE/results_BASE.csv`
- `outputs/fd002_BASE/best_pred_BASE.csv`

If your filenames differ, just edit the variables below.


In [ ]:
def first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    return None

# --- Where your FD002 model output artifacts live
OUTPUTS_DIR = PROJECT_ROOT / "outputs" / "fd002_BASE"   # <-- change if needed
CONFIG_JSON = first_existing([OUTPUTS_DIR / "config_BASE.json", PROJECT_ROOT / "config_BASE.json", Path("/content/config_BASE.json")])
RESULTS_CSV = first_existing([OUTPUTS_DIR / "results_BASE.csv", PROJECT_ROOT / "results_BASE.csv", Path("/content/results_BASE.csv")])
BEST_PRED_CSV = first_existing([OUTPUTS_DIR / "best_pred_BASE.csv", PROJECT_ROOT / "best_pred_BASE.csv", Path("/content/best_pred_BASE.csv")])

print("CONFIG_JSON   =", CONFIG_JSON)
print("RESULTS_CSV   =", RESULTS_CSV)
print("BEST_PRED_CSV =", BEST_PRED_CSV)

assert BEST_PRED_CSV is not None, "best_pred_BASE.csv not found. Put it under outputs/fd002_BASE/ or PROJECT_ROOT/."


## 3) Detect train/test CSV paths

We try in this order:
1) Paths from `config_BASE.json` (if present)
2) Common repo locations under `data/processed/.../FD002/`
3) `/content/*.csv` (if you uploaded directly to runtime)


In [ ]:
cfg = None
if CONFIG_JSON is not None:
    cfg = json.load(open(CONFIG_JSON, "r"))

candidates_train = []
candidates_test = []

if cfg is not None:
    # config may contain /content paths; we still try them first.
    if "TRAIN_PATH" in cfg: candidates_train.append(Path(cfg["TRAIN_PATH"]))
    if "TEST_PATH" in cfg:  candidates_test.append(Path(cfg["TEST_PATH"]))

# Common repo candidates (edit if your naming differs)
candidates_train += [
    PROJECT_ROOT / "data/processed/CMAPSS_norm_full/FD002/train_FD002_full_norm.csv",
    PROJECT_ROOT / "data/processed/CMAPSS_norm_full/FD002/train_FD002_full_global_zscore.csv",
    PROJECT_ROOT / "data/processed/CMAPSS_norm_full/FD002/train_FD002_full_norm.csv".replace("norm", "global_zscore"),
]
candidates_test += [
    PROJECT_ROOT / "data/processed/CMAPSS_norm_full/FD002/test_FD002_full_norm.csv",
    PROJECT_ROOT / "data/processed/CMAPSS_norm_full/FD002/test_FD002_full_global_zscore.csv",
    PROJECT_ROOT / "data/processed/CMAPSS_norm_full/FD002/test_FD002_full_norm.csv".replace("norm", "global_zscore"),
]

# /content fallback
candidates_train += [Path("/content/train_FD002_full_global_zscore.csv"), Path("/content/train_FD002_full_norm.csv")]
candidates_test  += [Path("/content/test_FD002_full_global_zscore.csv"),  Path("/content/test_FD002_full_norm.csv")]

TRAIN_CSV = first_existing(candidates_train)
TEST_CSV  = first_existing(candidates_test)

print("TRAIN_CSV =", TRAIN_CSV)
print("TEST_CSV  =", TEST_CSV)

assert TRAIN_CSV is not None and TEST_CSV is not None, (
    "Could not locate FD002 train/test CSV. "
    "Put them under data/processed/CMAPSS_norm_full/FD002/ or edit TRAIN_CSV/TEST_CSV above."
)


## 4) Load FD002 data + infer column names

We must have:
- `engine_id` (or `unit`/`unit_number`)
- `cycle` (or `time`)
- sensor columns (e.g. `os1..os3`, `s1..s21`, or similar)


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("train shape:", train_df.shape)
print("test  shape:", test_df.shape)
print("train cols:", list(train_df.columns)[:30])

def infer_col(df, candidates, kind):
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"Could not infer {kind} column. Tried {candidates}. Available: {list(df.columns)}")

ENGINE_CANDIDATES = ["engine_id", "unit", "unit_number", "id"]
CYCLE_CANDIDATES  = ["cycle", "time", "t"]

engine_col = infer_col(train_df, ENGINE_CANDIDATES, "engine_id")
cycle_col  = infer_col(train_df, CYCLE_CANDIDATES, "cycle")

print("engine_col =", engine_col)
print("cycle_col  =", cycle_col)

# Determine sensor columns
sensor_cols = None
if cfg is not None and "features_head" in cfg:
    # features_head usually includes cycle and sensors (not engine_id)
    fh = list(cfg["features_head"])
    sensor_cols = [c for c in fh if (c.startswith("s") or c.startswith("os") or c.startswith("setting"))]
    # remove cycle if present
    sensor_cols = [c for c in sensor_cols if c != cycle_col]
    # keep only if columns exist
    sensor_cols = [c for c in sensor_cols if c in train_df.columns]
    if len(sensor_cols) == 0:
        sensor_cols = None

if sensor_cols is None:
    # fallback heuristic
    sensor_cols = [c for c in train_df.columns if (c.startswith("s") or c.startswith("os") or c.startswith("setting"))]
    # avoid label-like cols
    sensor_cols = [c for c in sensor_cols if c.lower() not in ("rul", "label", "target")]

missing = [c for c in [engine_col, cycle_col] if c not in train_df.columns]
assert not missing, f"Missing required columns: {missing}"

print("sensor_cols (n=%d):" % len(sensor_cols), sensor_cols[:30])
assert len(sensor_cols) >= 5, "Sensor column detection seems wrong; please adjust sensor_cols."


## 5) Parameters

- `N_BASELINE`: number of first cycles per engine to define baseline.
- `MAPPING`: choose `'sigmoid'` or `'minmax'`.
- `MAP_FIT_ON`: `'train+test'` matches your friend's note (global dataset). For strict separation use `'train'`.
- `SMOOTH_WINDOW`: set to 0 to disable smoothing.


In [ ]:
N_BASELINE = 20
EPS = 1e-6

MAPPING = "sigmoid"      # "sigmoid" or "minmax"
MAP_FIT_ON = "train+test" # "train" or "train+test"

SMOOTH_WINDOW = 5         # 0 disables smoothing
TOPK_SENSORS = 3          # top contributors per row (|z|). Set 0 to disable.


## 6) Compute baseline deviation score (raw) + anomaly_score (0–1)

In [ ]:
def compute_baseline_stats(df, engine_col, cycle_col, sensor_cols, n_baseline=20, eps=1e-6):
    # Use first N cycles per engine (sorted by cycle).
    df_sorted = df.sort_values([engine_col, cycle_col], kind="mergesort")
    firstN = df_sorted.groupby(engine_col, as_index=False).head(n_baseline)

    stats = firstN.groupby(engine_col)[sensor_cols].agg(["mean", "std"])
    # flatten columns
    stats.columns = [f"{c}__{stat}" for c, stat in stats.columns]
    stats = stats.reset_index()

    # add eps to std later
    return stats

def attach_scores(df, stats, engine_col, cycle_col, sensor_cols, eps=1e-6):
    # Merge keeps left order
    out = df[[engine_col, cycle_col] + sensor_cols].merge(stats, on=engine_col, how="left", sort=False)

    mu = np.stack([out[f"{c}__mean"].to_numpy(dtype=float) for c in sensor_cols], axis=1)
    sd = np.stack([out[f"{c}__std"].to_numpy(dtype=float) for c in sensor_cols], axis=1) + eps
    x  = out[sensor_cols].to_numpy(dtype=float)

    z = (x - mu) / sd
    score_raw = np.sqrt(np.mean(z**2, axis=1))  # RMS z-score

    out2 = df[[engine_col, cycle_col]].copy()
    out2["score_raw"] = score_raw
    return out2, z  # z returned for optional explainability

baseline_stats = compute_baseline_stats(train_df, engine_col, cycle_col, sensor_cols, n_baseline=N_BASELINE, eps=EPS)

train_scores, _ = attach_scores(train_df, baseline_stats, engine_col, cycle_col, sensor_cols, eps=EPS)
test_scores,  test_z = attach_scores(test_df,  baseline_stats, engine_col, cycle_col, sensor_cols, eps=EPS)

print("train_scores:", train_scores.shape, "test_scores:", test_scores.shape)
print(train_scores.head())


## 7) Map `score_raw` -> `anomaly_score` (0–1) + optional smoothing

In [ ]:
if MAP_FIT_ON == "train+test":
    fit_scores = pd.concat([train_scores["score_raw"], test_scores["score_raw"]], axis=0)
else:
    fit_scores = train_scores["score_raw"]

s_min = float(fit_scores.min())
s_max = float(fit_scores.max())
s_med = float(fit_scores.median())

def map_minmax(s):
    return np.clip((s - s_min) / (s_max - s_min + 1e-12), 0.0, 1.0)

def map_sigmoid(s, k=1.0, c=None):
    if c is None:
        c = s_med
    return 1.0 / (1.0 + np.exp(-k * (s - c)))

train_scores["anomaly_minmax"]  = map_minmax(train_scores["score_raw"].to_numpy())
test_scores["anomaly_minmax"]   = map_minmax(test_scores["score_raw"].to_numpy())
train_scores["anomaly_sigmoid"] = map_sigmoid(train_scores["score_raw"].to_numpy(), k=1.0, c=s_med)
test_scores["anomaly_sigmoid"]  = map_sigmoid(test_scores["score_raw"].to_numpy(),  k=1.0, c=s_med)

chosen_col = "anomaly_sigmoid" if MAPPING == "sigmoid" else "anomaly_minmax"
train_scores["anomaly_score"] = train_scores[chosen_col]
test_scores["anomaly_score"]  = test_scores[chosen_col]

def smooth_per_engine(df_scores, engine_col, cycle_col, score_col, window):
    if window <= 1:
        return df_scores
    df_sorted = df_scores.sort_values([engine_col, cycle_col], kind="mergesort").copy()
    df_sorted["anomaly_score_smoothed"] = (
        df_sorted.groupby(engine_col)[score_col]
        .rolling(window=window, min_periods=1)
        .mean()
        .reset_index(level=0, drop=True)
    )
    # Restore original order for easier row-wise alignment with test_df
    return df_sorted.sort_index()

if SMOOTH_WINDOW and SMOOTH_WINDOW > 1:
    train_scores = smooth_per_engine(train_scores, engine_col, cycle_col, "anomaly_score", SMOOTH_WINDOW)
    test_scores  = smooth_per_engine(test_scores,  engine_col, cycle_col, "anomaly_score", SMOOTH_WINDOW)
else:
    train_scores["anomaly_score_smoothed"] = train_scores["anomaly_score"]
    test_scores["anomaly_score_smoothed"]  = test_scores["anomaly_score"]

train_scores.head(), test_scores.head()


## 8) Explainability (optional): top-K sensors by |z| per cycle

This can be helpful for decision-support UIs.


In [ ]:
def topk_sensors(abs_z, sensor_cols, k=3):
    if k <= 0:
        return None, None
    arr = abs_z
    n = arr.shape[0]
    k = min(k, arr.shape[1])
    idx = np.argpartition(arr, -k, axis=1)[:, -k:]
    # sort those k indices by descending value
    rows = np.arange(n)[:, None]
    vals_k = arr[rows, idx]
    order = np.argsort(vals_k, axis=1)[:, ::-1]
    idx_sorted = idx[rows, order]
    vals_sorted = arr[rows, idx_sorted]
    top_s = [[sensor_cols[j] for j in row] for row in idx_sorted]
    top_v = [[float(v) for v in row] for row in vals_sorted]
    return top_s, top_v

if TOPK_SENSORS and TOPK_SENSORS > 0:
    abs_z = np.abs(test_z)
    top_s, top_v = topk_sensors(abs_z, sensor_cols, k=TOPK_SENSORS)
    test_scores = test_scores.copy()
    test_scores["top_sensors"] = [";".join(s) for s in top_s]
    test_scores["top_abs_z"] = [";".join([f"{v:.4f}" for v in vs]) for vs in top_v]

test_scores.head()


## 9) Save outputs

Creates:
- `outputs/anomaly/FD002/fd002_anomaly_train.csv`
- `outputs/anomaly/FD002/fd002_anomaly_test.csv`


In [ ]:
OUT_DIR = PROJECT_ROOT / "outputs" / "anomaly" / "FD002"
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_out = OUT_DIR / "fd002_anomaly_train.csv"
test_out  = OUT_DIR / "fd002_anomaly_test.csv"

train_scores.to_csv(train_out, index=False)
test_scores.to_csv(test_out, index=False)

print("[OK] wrote:", train_out)
print("[OK] wrote:", test_out)


## 10) Decision-support table: merge anomaly score with model predictions (BASE)

We expect `best_pred_BASE.csv` to contain at least `y_true` and `y_pred`.
If it doesn't contain `engine_id`/`cycle`, we attach them from `test_df` **by row order** (must match).


In [ ]:
preds = pd.read_csv(BEST_PRED_CSV)
print("preds shape:", preds.shape, "cols:", preds.columns.tolist())

# Attach engine_id + cycle if missing
if engine_col not in preds.columns or cycle_col not in preds.columns:
    assert len(preds) == len(test_df), (
        f"Row count mismatch: preds={len(preds)} vs test_df={len(test_df)}. "
        "Cannot safely attach engine_id/cycle. Ensure BEST_PRED_CSV aligns with TEST_CSV."
    )
    preds = pd.concat([
        test_df[[engine_col, cycle_col]].reset_index(drop=True),
        preds.reset_index(drop=True)
    ], axis=1)

# Merge anomaly scores
# (Use smoothed score by default)
ds = preds.merge(
    test_scores[[engine_col, cycle_col, "anomaly_score", "anomaly_score_smoothed"] + (["top_sensors", "top_abs_z"] if "top_sensors" in test_scores.columns else [])],
    on=[engine_col, cycle_col],
    how="left",
    sort=False
)

print("decision-support shape:", ds.shape)
ds.head()


## 11) Save decision-support table

In [ ]:
ds_out = OUT_DIR / "fd002_decision_support_BASE.csv"
ds.to_csv(ds_out, index=False)
print("[OK] wrote:", ds_out)


## 12) Quick sanity checks

In [ ]:
print("anomaly_score range:", float(ds["anomaly_score"].min()), float(ds["anomaly_score"].max()))
print("anomaly_score_smoothed range:", float(ds["anomaly_score_smoothed"].min()), float(ds["anomaly_score_smoothed"].max()))

# show a few most-anomalous rows
ds.sort_values("anomaly_score_smoothed", ascending=False).head(10)
